In [1]:
import os
import pandas as pd

N_CLASSES = 8
VAL_N = 200            # per class
TEST_N = 200           # per class
HOLDOUT_SEED = 0       # dedicated seed for the (fixed) val/test holdout
MC_SEEDS = [1, 2, 3, 4, 5]   # the five Monte Carlo runs — documented in methods
 
SCENE_TRAIN = {0: 69567, 1: 1000, 2: 500, 3: 100, 4: 50, 5: 30, 6: 10}
CSV_DIR = "CSV_Files"

# Val/Test extract

In [2]:
val_frames, test_frames = [], []
remainder_pools = {}   # class -> rows NOT in val/test (safe to train on)

# val/test set
for i in range(N_CLASSES):
    df = pd.read_csv(f"{CSV_DIR}/F{i}M.csv")
    df["Labels"] = i
 
    # Deterministic shuffle of the whole class with the dedicated holdout seed
    shuffled = df.sample(frac=1.0, random_state=HOLDOUT_SEED).reset_index(drop=True)
 
    # Firewall: carve val/test off the top, before anything else touches the data
    val_frames.append(shuffled.iloc[:VAL_N])
    test_frames.append(shuffled.iloc[VAL_N:VAL_N + TEST_N])
 
    # Everything else is the training-eligible pool for this class
    remainder_pools[i] = shuffled.iloc[VAL_N + TEST_N:].reset_index(drop=True)
 
os.makedirs(f"{CSV_DIR}/eval", exist_ok=True)
pd.concat(val_frames, ignore_index=True).to_csv(f"{CSV_DIR}/eval/val.csv", index=False)
pd.concat(test_frames, ignore_index=True).to_csv(f"{CSV_DIR}/eval/test.csv", index=False)



# Scenarios train sets

In [3]:
for seed in MC_SEEDS:
    # Shuffle each class's remainder ONCE per seed, then slice scenes from it.
    # Slicing from one shuffle makes scenes nested: scene6 ⊂ scene5 ⊂ ... ⊂ scene0.
    per_class_shuffled = {
        i: remainder_pools[i].sample(frac=1.0, random_state=seed).reset_index(drop=True)
        for i in range(N_CLASSES)
    }
 
    for scene, n_train in SCENE_TRAIN.items():
        train_frames = [
            per_class_shuffled[i].iloc[:n_train] for i in range(N_CLASSES)
        ]
        df_train = pd.concat(train_frames, ignore_index=True)
 
        out_dir = f"{CSV_DIR}/seed{seed}/scene{scene}"
        os.makedirs(out_dir, exist_ok=True)
        df_train.to_csv(f"{out_dir}/train.csv", index=False)


In [4]:
print("Done. Layout:")
print(f"  {CSV_DIR}/eval/val.csv, test.csv        (fixed, shared across seeds)")
print(f"  {CSV_DIR}/seed<1-5>/scene<0-6>/train.csv (varies per seed)")


Done. Layout:
  CSV_Files/eval/val.csv, test.csv        (fixed, shared across seeds)
  CSV_Files/seed<1-5>/scene<0-6>/train.csv (varies per seed)
